# Phase 2C — 2026 MLB 독립 재학습 및 2025 대비 동일 프로토콜 비교

**목표**: 2025년과 **동일한 22개 Pre-game / 8개 In-game 입력변수**, DNN 구조, 검증 기반 조기 종료 및 머신러닝 비교 모델로 **2026년을 처음부터 독립적으로 학습**합니다. 이는 2025년 모델을 2026년에 적용한 *외부 검증*과 구별됩니다.

- 업로드 **2개**: `phase2b_2026_external_output.zip` 및 `phase1e_baseline_output.zip`.
- 2026 학습/검증/테스트는 서로 다른 경기와 **서로 다른 날짜**를 사용하며 동일 날짜가 겹치지 않습니다.
- 전처리 중앙값·StandardScaler·머신러닝 학습은 2026년 **train만** 사용합니다.
- 검증셋으로 조기 종료/비교 모델 후보 선택, 테스트셋은 **선택 이후 한 번만** 평가합니다.
- Pre-game: 2025 최종 C1 구조(128→64→32, Dropout .15, L2 .0001), 학습률 .001, batch 64, patience 14.
- In-game: 2025 구조(64→32→16, Dropout .20, L2 없음), 학습률 .001, batch 128, patience 12; **경기별 동일 가중치**.
- 일반 비교모델: Logistic Regression, Random Forest, Histogram Gradient Boosting, XGBoost. In-game은 경험적 승률·리드 팀 규칙도 평가.

**주의**: 2026년 게임 기록에서 *실제 선발투수*를 사용한 경우 경기 전 정보 가용성에 제한이 있으며, 2026년 테스트셋은 한 시즌의 9월 경기이므로 2025년 9월과 직접적인 통제 비교는 아닙니다.

In [ ]:
!pip -q install "xgboost>=2.0,<4" tensorflow scikit-learn pandas matplotlib

In [ ]:
import io, json, os, random, zipfile, warnings
from pathlib import Path
import numpy as np, pandas as pd, tensorflow as tf
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, brier_score_loss, log_loss
from sklearn.calibration import calibration_curve
from xgboost import XGBClassifier
from google.colab import files
import matplotlib.pyplot as plt

SEED=2026
random.seed(SEED); np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)
OUT=Path('/content/phase2c_2026_retraining_output'); OUT.mkdir(parents=True,exist_ok=True)
print('두 ZIP 파일을 업로드하세요: phase2b_2026_external_output.zip, phase1e_baseline_output.zip')
_ = files.upload()
required=['phase2b_2026_external_output.zip','phase1e_baseline_output.zip']
assert all(Path(x).is_file() for x in required),f'필수 파일 확인: {required}'
z26=zipfile.ZipFile(required[0]); z25=zipfile.ZipFile(required[1])
def csv(z,name): return pd.read_csv(io.BytesIO(z.read(name)))
pre=csv(z26,'2026_pregame_features.csv')
ing=csv(z26,'2026_ingame_predictions.csv').drop(columns=['prob_ingame_dnn'])
old_pre=csv(z25,'pregame_comparison_metrics.csv')
old_ing=csv(z25,'ingame_comparison_metrics.csv')
print('2026 games',len(pre),'snapshots',len(ing))
assert len(pre)==2429 and pre.gamePk.nunique()==2429
assert len(ing)==43204 and ing.gamePk.nunique()==2429
assert not ing.duplicated(['gamePk','inning','is_bottom']).any()
assert set(pre.gamePk)==set(ing.gamePk)
assert pre.target.isin([0,1]).all() and ing.target.isin([0,1]).all()

## 1. 날짜 기반 독립 분할 및 입력값 검사

In [ ]:
pre['date']=pd.to_datetime(pre.date); ing['date']=pd.to_datetime(ing.date)
dates=np.array(sorted(pre.date.unique()))
ntr=int(len(dates)*.70); nval=int(len(dates)*.15)
trdates=dates[:ntr]; vdates=dates[ntr:ntr+nval]; tedates=dates[ntr+nval:]
assert len(trdates) and len(vdates) and len(tedates)
assign={'train':trdates,'validation':vdates,'test':tedates}
for d in (pre,ing):
    d['split']=''
    for name,days in assign.items():d.loc[d.date.isin(days),'split']=name
    assert d.split.ne('').all()
assert not(set(trdates)&set(vdates)) and not(set(trdates)&set(tedates))
assert (pre.groupby('gamePk').split.nunique()==1).all()
assert (ing.groupby('gamePk').split.nunique()==1).all()
assert ing.groupby('gamePk').target.nunique().eq(1).all()
assert ing.groupby('gamePk').split.nunique().eq(1).all()
partition=pre[['gamePk','date','target','split']].sort_values(['date','gamePk']).copy()
partition.to_csv(OUT/'2026_game_partitions.csv',index=False)
print('분할 경계 (날짜 수는 70/15/15, 경기 수는 정확히 비례하지 않음):')
for name in ['train','validation','test']:
    p=pre.loc[pre.split==name]
    print(name,'dates',p.date.min().date(),p.date.max().date(),'games',len(p),'snapshots',len(ing.loc[ing.split==name]))

PRE=['home_win_pct','away_win_pct','win_pct_diff','home_run_diff','away_run_diff','run_diff_diff',
     'home_runs_per_game','away_runs_per_game','home_ra_per_game','away_ra_per_game',
     'home_recent_form','away_recent_form','form_diff','home_sp_era','away_sp_era','sp_era_diff',
     'home_sp_whip','away_sp_whip','sp_whip_diff','home_sp_k9','away_sp_k9','sp_k9_diff']
ING=['inning','is_bottom','score_diff','home_runs','away_runs','home_hits','away_hits','abs_score_diff']
assert all(f in pre for f in PRE) and all(f in ing for f in ING)
assert (ing.home_runs-ing.away_runs == ing.score_diff).all()
assert (ing.score_diff.abs()==ing.abs_score_diff).all()
assert ing['snapshot'].eq('pre_half_inning').all()


## 2. Pre-game 시즌 초 선발투수 결측: 2026년 학습셋 중앙값만 이용

2026 외부 검증 때 적용된 **2025년 기준 결측 대체값**은 여기서 그대로 사용하지 않습니다. 소스가 `prior_game_logs`가 아닌 값들을 결측으로 되돌리고, 2026년 train에서 구한 중앙값으로 대체한 뒤 차이 변수를 재계산합니다. 원본 2025년과 동일한 0.5 승률 대체 규칙은 유지합니다.

In [ ]:
pre=pre.copy()
impute_report={}
for stat in ['era','whip','k9']:
    colh=f'home_sp_{stat}'; cola=f'away_sp_{stat}'
    src_h=pre.home_sp_source.eq('prior_game_logs')
    src_a=pre.away_sp_source.eq('prior_game_logs')
    trainpool=pd.concat([pre.loc[(pre.split=='train')&src_h,colh],pre.loc[(pre.split=='train')&src_a,cola]],ignore_index=True)
    med=float(trainpool.dropna().median())
    assert np.isfinite(med)
    impute_report[stat]={'train_only_pooled_median':med,'home_missing_sources':int((~src_h).sum()),'away_missing_sources':int((~src_a).sum())}
    pre.loc[~src_h,colh]=med
    pre.loc[~src_a,cola]=med
    pre[f'sp_{stat}_diff']=pre[colh]-pre[cola]   # 홈 - 원정; 2025와 동일
assert pre[PRE].notna().all().all() and ing[ING].notna().all().all()
print('train-only pitcher medians:',impute_report)
print('Early win% defaults:',int((pre.home_winpct_imputed.astype(bool)|pre.away_winpct_imputed.astype(bool)).sum()))
pre.to_csv(OUT/'2026_pregame_features_training_cleaned.csv',index=False)


## 3. 평가 함수 및 공통 머신러닝 후보

통일 비교는 **2025 Phase 1E와 동일한 후보군과 검증셋 Brier Score 선택 규칙**을 사용합니다.

In [ ]:
def evaluate(y,p):
    y=np.asarray(y,int);p=np.clip(np.asarray(p,float).ravel(),1e-7,1-1e-7)
    b=np.clip(np.digitize(p,np.linspace(0,1,11)[1:-1],right=True),0,9)
    ece=sum((b==k).mean()*abs(y[b==k].mean()-p[b==k].mean()) for k in range(10) if (b==k).any())
    return dict(n=len(y),accuracy=float(accuracy_score(y,p>=.5)),auc=float(roc_auc_score(y,p)) if len(np.unique(y))==2 else float('nan'),
                brier=float(brier_score_loss(y,p)),logloss=float(log_loss(y,p,labels=[0,1])),ece10=float(ece))

def candidates():
    return {
      'Random Forest':[
        ('RF_leaf5',RandomForestClassifier(n_estimators=250,min_samples_leaf=5,max_features='sqrt',n_jobs=-1,random_state=SEED)),
        ('RF_leaf15',RandomForestClassifier(n_estimators=250,min_samples_leaf=15,max_features='sqrt',n_jobs=-1,random_state=SEED))],
      'Histogram Gradient Boosting':[
        ('HGB_leaf20',HistGradientBoostingClassifier(max_iter=150,learning_rate=.05,max_leaf_nodes=15,min_samples_leaf=20,l2_regularization=1.,random_state=SEED)),
        ('HGB_leaf50',HistGradientBoostingClassifier(max_iter=150,learning_rate=.05,max_leaf_nodes=15,min_samples_leaf=50,l2_regularization=1.,random_state=SEED))],
      'XGBoost':[
        ('XGB_depth2',XGBClassifier(n_estimators=220,max_depth=2,learning_rate=.03,subsample=.85,colsample_bytree=.9,reg_lambda=5.,objective='binary:logistic',eval_metric='logloss',tree_method='hist',n_jobs=2,random_state=SEED)),
        ('XGB_depth3',XGBClassifier(n_estimators=220,max_depth=3,learning_rate=.03,subsample=.85,colsample_bytree=.9,reg_lambda=5.,objective='binary:logistic',eval_metric='logloss',tree_method='hist',n_jobs=2,random_state=SEED))]
    }
def fit_baselines(df,features,label):
    a=df[df.split=='train'];v=df[df.split=='validation'];t=df[df.split=='test']
    x=a[features].to_numpy(float);xv=v[features].to_numpy(float);xt=t[features].to_numpy(float)
    yy=a.target.to_numpy(int);yv=v.target.to_numpy(int)
    s=StandardScaler().fit(x)
    lr=LogisticRegression(C=1.,max_iter=2500,random_state=SEED).fit(s.transform(x),yy)
    pr={'Logistic Regression':lr.predict_proba(s.transform(xt))[:,1]}
    vals=[dict(task=label,model='Logistic Regression',candidate='LR_fixed',**evaluate(yv,lr.predict_proba(s.transform(xv))[:,1]))]
    selected={}
    for typ,variants in candidates().items():
        results=[]
        for tag,m in variants:
            m.fit(x,yy)
            pv=m.predict_proba(xv)[:,1]; score=evaluate(yv,pv)
            vals.append(dict(task=label,model=typ,candidate=tag,**score))
            results.append((score['brier'],-score['auc'],tag,m))
        best=min(results,key=lambda x:(x[0],x[1]))
        selected[typ]=best[2]
        pr[typ]=best[3].predict_proba(xt)[:,1]
        print(label,typ,'chosen',best[2],'valid Brier',round(best[0],6))
    return pr,pd.DataFrame(vals),selected


## 4. 2026 Pre-game DNN C1 재학습 (하이퍼파라미터 재탐색 없음)

In [ ]:
train=pre[pre.split=='train']; val=pre[pre.split=='validation']; test=pre[pre.split=='test']
pre_scaler=StandardScaler().fit(train[PRE].to_numpy(float))
xtr=pre_scaler.transform(train[PRE].to_numpy(float)).astype('float32')
xv=pre_scaler.transform(val[PRE].to_numpy(float)).astype('float32')
xt=pre_scaler.transform(test[PRE].to_numpy(float)).astype('float32')
ytr=train.target.to_numpy(int);yv=val.target.to_numpy(int);yte=test.target.to_numpy(int)
tf.keras.backend.clear_session();tf.keras.utils.set_random_seed(SEED)
from tensorflow.keras import regularizers
pre_model=tf.keras.Sequential([tf.keras.Input(shape=(22,)),
    tf.keras.layers.Dense(128,activation='relu',kernel_regularizer=regularizers.l2(.0001)),
    tf.keras.layers.Dropout(.15),
    tf.keras.layers.Dense(64,activation='relu',kernel_regularizer=regularizers.l2(.0001)),
    tf.keras.layers.Dropout(.15),
    tf.keras.layers.Dense(32,activation='relu',kernel_regularizer=regularizers.l2(.0001)),
    tf.keras.layers.Dense(1,activation='sigmoid')])
pre_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=.001),loss='binary_crossentropy',metrics=['accuracy'])
pre_hist=pre_model.fit(xtr,ytr,validation_data=(xv,yv),epochs=180,batch_size=64,verbose=0,
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=14,restore_best_weights=True)])
pre_best=int(np.argmin(pre_hist.history['val_loss'])+1)
print('Pre-game best epoch',pre_best,'total epochs',len(pre_hist.history['loss']))
pre_p=pre_model.predict(xt,verbose=0).ravel()
print('Pre-game DNN TEST:',evaluate(yte,pre_p))
pre_model.save(OUT/'2026_pregame_DNN.keras')


## 5. 2026 In-game DNN 재학습 (경기별 동일 가중치)

In [ ]:
train_i=ing[ing.split=='train'];val_i=ing[ing.split=='validation'];test_i=ing[ing.split=='test']
in_scaler=StandardScaler().fit(train_i[ING].to_numpy(float))
xi=in_scaler.transform(train_i[ING].to_numpy(float)).astype('float32')
xiv=in_scaler.transform(val_i[ING].to_numpy(float)).astype('float32')
xit=in_scaler.transform(test_i[ING].to_numpy(float)).astype('float32')
yi=train_i.target.to_numpy(int);yiv=val_i.target.to_numpy(int);yit=test_i.target.to_numpy(int)
w=1./train_i.groupby('gamePk').gamePk.transform('size').to_numpy(float)
w=w/np.mean(w)
tf.keras.backend.clear_session();tf.keras.utils.set_random_seed(SEED)
ingame_model=tf.keras.Sequential([tf.keras.Input(shape=(8,)),
 tf.keras.layers.Dense(64,activation='relu'),tf.keras.layers.Dropout(.2),
 tf.keras.layers.Dense(32,activation='relu'),tf.keras.layers.Dropout(.2),
 tf.keras.layers.Dense(16,activation='relu'),tf.keras.layers.Dense(1,activation='sigmoid')])
ingame_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=.001),loss='binary_crossentropy',metrics=['accuracy'])
ingame_hist=ingame_model.fit(xi,yi,sample_weight=w,validation_data=(xiv,yiv),epochs=200,batch_size=128,verbose=0,
 callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=12,restore_best_weights=True)])
in_best=int(np.argmin(ingame_hist.history['val_loss'])+1)
print('In-game best epoch',in_best,'total epochs',len(ingame_hist.history['loss']))
in_p=ingame_model.predict(xit,verbose=0).ravel()
print('In-game DNN TEST:',evaluate(yit,in_p))
ingame_model.save(OUT/'2026_ingame_DNN.keras')


## 6. 일반 베이스라인 및 야구 상황 기반 베이스라인

In [ ]:
pre_bl,pre_val_models,pre_chosen=fit_baselines(pre,PRE,'Pre-game')
in_bl,in_val_models,in_chosen=fit_baselines(ing,ING,'In-game')
# 2025와 동일한 경험적 승률: inning × is_bottom × score_diff capped [-5,+5], alpha=30
exp=train_i.copy();exp['score_bin']=exp.score_diff.clip(-5,5)
prior=float(np.dot(exp.target.to_numpy(int),w)/w.sum())
exp['w']=w;exp['wy']=w*exp.target
cells=exp.groupby(['inning','is_bottom','score_bin']).agg(n=('w','sum'),wins=('wy','sum'))
def empirical(d):
    keys=list(zip(d.inning.astype(int),d.is_bottom.astype(int),d.score_diff.clip(-5,5).astype(int)))
    return np.asarray([(float(cells.loc[k,'wins'])+30*prior)/(float(cells.loc[k,'n'])+30) if k in cells.index else prior for k in keys])
def leading_rule(d):
    return np.where(d.score_diff.to_numpy()>0,1.,np.where(d.score_diff.to_numpy()<0,0.,prior))

pre_result=test[['gamePk','date','target']].reset_index(drop=True).copy();pre_result['DNN']=pre_p
for name,pred in pre_bl.items():pre_result[name]=pred
in_result=test_i[['gamePk','date','target','inning','is_bottom','score_diff']].reset_index(drop=True).copy()
in_result['DNN']=in_p
for name,pred in in_bl.items():in_result[name]=pred
in_result['Empirical Win Expectancy']=empirical(test_i)
in_result['Leading-Team Rule']=leading_rule(test_i)
pre_names=['DNN','Logistic Regression','Random Forest','Histogram Gradient Boosting','XGBoost']
in_names=pre_names+['Empirical Win Expectancy','Leading-Team Rule']
pre_metrics=pd.DataFrame([{'model':name,**evaluate(pre_result.target,pre_result[name])} for name in pre_names])
in_metrics=pd.DataFrame([{'model':name,**evaluate(in_result.target,in_result[name])} for name in in_names])
print('PRE-GAME\n',pre_metrics.to_string(index=False))
print('IN-GAME\n',in_metrics.to_string(index=False))


## 7. 이닝별 성능, 2025–2026 비교, 검증 및 출력

In [ ]:
by_inning=pd.DataFrame([dict(inning=int(inn),model=name,**evaluate(g.target,g[name]))
    for inn,g in in_result.groupby('inning') for name in in_names])
old_pre['season']='2025';old_ing['season']='2025';pre_metrics['season']='2026';in_metrics['season']='2026'
pre_cmp=pd.concat([old_pre,pre_metrics],ignore_index=True)
in_cmp=pd.concat([old_ing,in_metrics],ignore_index=True)
pre_cmp['task']='Pre-game';in_cmp['task']='In-game'
comparison=pd.concat([pre_cmp,in_cmp],ignore_index=True)
# 확인: 같은 시점의 2026 test로만 비교; 외부검증 2026 전체시즌 지표와 혼동하지 않음
assert pre_result.gamePk.nunique()==pre_result.shape[0]
assert in_result.gamePk.nunique()==test_i.gamePk.nunique()
assert len(pre_result)==(pre.split=='test').sum() and len(in_result)==(ing.split=='test').sum()
assert np.isfinite(pre_result[pre_names].to_numpy()).all() and np.isfinite(in_result[in_names].to_numpy()).all()
pre_result.to_csv(OUT/'2026_pregame_test_predictions.csv',index=False)
in_result.to_csv(OUT/'2026_ingame_test_predictions.csv',index=False)
pre_metrics.to_csv(OUT/'2026_pregame_test_metrics.csv',index=False)
in_metrics.to_csv(OUT/'2026_ingame_test_metrics.csv',index=False)
by_inning.to_csv(OUT/'2026_ingame_by_inning.csv',index=False)
comparison.to_csv(OUT/'2025_2026_same_protocol_comparison.csv',index=False)
pd.concat([pre_val_models,in_val_models]).to_csv(OUT/'2026_validation_baseline_candidates.csv',index=False)
for task,m,h,s,feat in [('pregame',pre_model,pre_hist,pre_scaler,PRE),('ingame',ingame_model,ingame_hist,in_scaler,ING)]:
    export={'year':2026,'features':feat,'scaler':{'mean':s.mean_.tolist(),'scale':s.scale_.tolist()},
       'layers':[{'W':a.tolist(),'b':b.tolist()} for a,b in (l.get_weights() for l in m.layers if isinstance(l,tf.keras.layers.Dense))]}
    (OUT/f'2026_{task}_model_export.json').write_text(json.dumps(export,ensure_ascii=False))
report={'design':'2026 independent within-season retraining, date-grouped 70/15/15',
 'caveat':'2026 actual starting pitcher availability before first pitch not verified',
 'season':2026,'games':len(pre),'snapshots':len(ing),
 'split_game_counts':pre.groupby('split').size().to_dict(),
 'split_snapshot_counts':ing.groupby('split').size().to_dict(),
 'train_only_pitcher_imputation':impute_report,'pregame_best_epoch':pre_best,'ingame_best_epoch':in_best,
 'pregame_architecture':[22,128,64,32,1],'ingame_architecture':[8,64,32,16,1],
 'baseline_selected':{'pregame':pre_chosen,'ingame':in_chosen},
 'classification_threshold':.5,'score_selection':'validation Brier, DNN early stop validation loss',
 'note':'2025 test covers September 2025; 2026 test covers a different date set. External season validation is a separate analysis.'}
(OUT/'2026_retraining_audit.json').write_text(json.dumps(report,ensure_ascii=False,indent=2))
fig,axes=plt.subplots(1,2,figsize=(12,4))
for task,y,p,ax in [('Pre-game',pre_result.target,pre_result.DNN,axes[0]),('In-game',in_result.target,in_result.DNN,axes[1])]:
    obs,mean=calibration_curve(y,p,n_bins=10,strategy='uniform')
    ax.plot([0,1],[0,1],'--',color='gray');ax.plot(mean,obs,'o-');ax.set(xlabel='Predicted probability',ylabel='Observed home wins',title=task+' 2026 Test Calibration',xlim=(0,1),ylim=(0,1))
plt.tight_layout();fig.savefig(OUT/'2026_DNN_calibration.png',dpi=160);plt.show()
print('\n2025 vs 2026 same-protocol performance (NOT cross-season frozen-model evaluation):')
print(comparison[['season','task','model','n','accuracy','auc','brier']].to_string(index=False))


## 8. ZIP 생성 및 다운로드

In [ ]:
import shutil
archive=shutil.make_archive('/content/phase2c_2026_retraining_output','zip',root_dir=OUT)
print('OUTPUT',archive,'SIZE_MB',round(Path(archive).stat().st_size/1e6,2))
files.download(archive)
